fund_master cleaning

In [4]:
import pandas as pd

file_name = "/content/01_fund_master.csv"
df_master = pd.read_csv(file_name)

print("Before Cleaning:")
print(df_master.info())

str_cols = df_master.select_dtypes(include=['object']).columns
for col in str_cols:
    df_master[col] = df_master[col].astype(str).str.strip()

df_master['launch_date'] = pd.to_datetime(df_master['launch_date'], errors='coerce')

df_master.drop_duplicates(inplace=True)

df_master.sort_values(by='amfi_code', inplace=True)
df_master.reset_index(drop=True, inplace=True)

cleaned_file_name = "cleaned_01_fund_master.csv"
df_master.to_csv(cleaned_file_name, index=False)

print("\nAfter Cleaning:")
print(df_master.info())
print(f"Cleaned file saved as: {cleaned_file_name}")

Before Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   fund_house          40 non-null     object 
 2   scheme_name         40 non-null     object 
 3   category            40 non-null     object 
 4   sub_category        40 non-null     object 
 5   plan                40 non-null     object 
 6   launch_date         40 non-null     object 
 7   benchmark           40 non-null     object 
 8   expense_ratio_pct   40 non-null     float64
 9   exit_load_pct       40 non-null     float64
 10  min_sip_amount      40 non-null     int64  
 11  min_lumpsum_amount  40 non-null     int64  
 12  fund_manager        40 non-null     object 
 13  risk_category       40 non-null     object 
 14  sebi_category_code  40 non-null     object 
dtypes: float64(2), int64(3), object(10)
memory

nav_history cleaning

In [1]:
import pandas as pd

# Load file
df = pd.read_csv("/content/02_nav_history.csv")

print("Original Shape:", df.shape)

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.dropna(subset=["date"])

#removing duplicates
before_dup = len(df)
df = df.drop_duplicates(subset=["amfi_code", "date"], keep="last")
after_dup = len(df)

print(f"Duplicates Removed: {before_dup - after_dup}")

# Sorting by AMFI code and date
df = df.sort_values(["amfi_code", "date"])

# Forward-fill missing NAVs (within each fund)
df["nav"] = (
    df.groupby("amfi_code")["nav"]
      .transform(lambda x: x.ffill())
)

#Validating NAV > 0
invalid_nav = df[df["nav"] <= 0]

print(f"Rows with NAV <= 0: {len(invalid_nav)}")

df = df[df["nav"] > 0]



print("\nFinal Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDate Range:")
print(df["date"].min(), "to", df["date"].max())

print("\nUnique Funds:", df["amfi_code"].nunique())

output_file = "/content/nav_history_cleaned.csv"
df.to_csv(output_file, index=False)

print(f"\nCleaned file saved as: {output_file}")

df.head()

Original Shape: (46000, 3)
Duplicates Removed: 0
Rows with NAV <= 0: 0

Final Shape: (46000, 3)

Missing Values:
amfi_code    0
date         0
nav          0
dtype: int64

Date Range:
2022-01-03 00:00:00 to 2026-05-29 00:00:00

Unique Funds: 40

Cleaned file saved as: /content/nav_history_cleaned.csv


,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


aum_by_fund cleaning

In [5]:
import pandas as pd

file_name = "/content/03_aum_by_fund_house.csv"
df_aum = pd.read_csv(file_name)

print("Before Cleaning:")
print(df_aum.info())

str_cols = df_aum.select_dtypes(include=['object']).columns
for col in str_cols:
    df_aum[col] = df_aum[col].astype(str).str.strip()

df_aum['date'] = pd.to_datetime(df_aum['date'], errors='coerce')

df_aum.drop_duplicates(inplace=True)

df_aum.sort_values(by=['date', 'fund_house'], inplace=True)
df_aum.reset_index(drop=True, inplace=True)

cleaned_file_name = "cleaned_03_aum_by_fund_house.csv"
df_aum.to_csv(cleaned_file_name, index=False)

print("\nAfter Cleaning:")
print(df_aum.info())
print(f"Cleaned file saved as: {cleaned_file_name}")

Before Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            90 non-null     object 
 1   fund_house      90 non-null     object 
 2   aum_lakh_crore  90 non-null     float64
 3   aum_crore       90 non-null     int64  
 4   num_schemes     90 non-null     int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 3.6+ KB
None

After Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            90 non-null     datetime64[ns]
 1   fund_house      90 non-null     object        
 2   aum_lakh_crore  90 non-null     float64       
 3   aum_crore       90 non-null     int64         
 4   num_schemes     90 non-null     int64         
dtypes: datetime

monthly_sip_inflows cleaning

In [6]:
import pandas as pd

file_name = "/content/04_monthly_sip_inflows.csv"
df_sip = pd.read_csv(file_name)

print("Before Cleaning:")
print(df_sip.info())

df_sip['month'] = pd.to_datetime(df_sip['month'], format='%Y-%m', errors='coerce')

df_sip['yoy_growth_pct'] = df_sip['yoy_growth_pct'].fillna(0.0).astype(float)

df_sip.drop_duplicates(inplace=True)

df_sip.sort_values(by='month', inplace=True)
df_sip.reset_index(drop=True, inplace=True)

cleaned_file_name = "cleaned_04_monthly_sip_inflows.csv"
df_sip.to_csv(cleaned_file_name, index=False)

print("\nAfter Cleaning:")
print(df_sip.info())
print(f"Cleaned file saved as: {cleaned_file_name}")

Before Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   month                      48 non-null     object 
 1   sip_inflow_crore           48 non-null     int64  
 2   active_sip_accounts_crore  48 non-null     float64
 3   new_sip_accounts_lakh      48 non-null     float64
 4   sip_aum_lakh_crore         48 non-null     float64
 5   yoy_growth_pct             36 non-null     float64
dtypes: float64(4), int64(1), object(1)
memory usage: 2.4+ KB
None

After Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   month                      48 non-null     datetime64[ns]
 1   sip_inflow_crore           48 non-null     int64     

category_inflows cleaning

In [7]:
import pandas as pd

file_name = "/content/05_category_inflows.csv"
df_category = pd.read_csv(file_name)

print("Before Cleaning:")
print(df_category.info())

str_cols = df_category.select_dtypes(include=['object']).columns
for col in str_cols:
    df_category[col] = df_category[col].astype(str).str.strip()

df_category['month'] = pd.to_datetime(df_category['month'], format='%Y-%m', errors='coerce')

df_category.drop_duplicates(inplace=True)

df_category.sort_values(by=['month', 'category'], inplace=True)
df_category.reset_index(drop=True, inplace=True)

cleaned_file_name = "cleaned_05_category_inflows.csv"
df_category.to_csv(cleaned_file_name, index=False)

print("\nAfter Cleaning:")
print(df_category.info())
print(f"Cleaned file saved as: {cleaned_file_name}")

Before Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   month             144 non-null    object 
 1   category          144 non-null    object 
 2   net_inflow_crore  144 non-null    float64
dtypes: float64(1), object(2)
memory usage: 3.5+ KB
None

After Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   month             144 non-null    datetime64[ns]
 1   category          144 non-null    object        
 2   net_inflow_crore  144 non-null    float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 3.5+ KB
None
Cleaned file saved as: cleaned_05_category_inflows.csv


industry_folio_count cleaning

In [8]:
import pandas as pd

file_name = "/content/06_industry_folio_count.csv"
df_folio = pd.read_csv(file_name)

print("Before Cleaning:")
print(df_folio.info())

df_folio['month'] = pd.to_datetime(df_folio['month'], format='%Y-%m', errors='coerce')

df_folio.drop_duplicates(inplace=True)

df_folio.sort_values(by='month', inplace=True)
df_folio.reset_index(drop=True, inplace=True)

cleaned_file_name = "cleaned_06_industry_folio_count.csv"
df_folio.to_csv(cleaned_file_name, index=False)

print("\nAfter Cleaning:")
print(df_folio.info())
print(f"Cleaned file saved as: {cleaned_file_name}")

Before Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                21 non-null     object 
 1   total_folios_crore   21 non-null     float64
 2   equity_folios_crore  21 non-null     float64
 3   debt_folios_crore    21 non-null     float64
 4   hybrid_folios_crore  21 non-null     float64
 5   others_folios_crore  21 non-null     float64
dtypes: float64(5), object(1)
memory usage: 1.1+ KB
None

After Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   month                21 non-null     datetime64[ns]
 1   total_folios_crore   21 non-null     float64       
 2   equity_folios_crore  21 non-null     float64       
 3   debt_folios_cro

investor_transaction cleaning

In [2]:
import pandas as pd

# Load file
df = pd.read_csv("/content/08_investor_transactions.csv")

print("Original Shape:", df.shape)

df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

invalid_dates = df["transaction_date"].isna().sum()
print(f"Invalid Dates: {invalid_dates}")

df = df.dropna(subset=["transaction_date"])

df["transaction_type"] = (
    df["transaction_type"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Mapping common variants
transaction_mapping = {
    "Sip": "SIP",
    "Lumpsum": "Lumpsum",
    "Redemption": "Redemption",
    "Redeem": "Redemption",
    "Redeemed": "Redemption"
}

df["transaction_type"] = df["transaction_type"].replace(transaction_mapping)

valid_transaction_types = ["SIP", "Lumpsum", "Redemption"]

invalid_txn = df[
    ~df["transaction_type"].isin(valid_transaction_types)
]

print("\nInvalid Transaction Types:")
print(invalid_txn["transaction_type"].value_counts())



invalid_amounts = (df["amount_inr"] <= 0).sum()

print(f"\nTransactions with amount <= 0: {invalid_amounts}")


df = df[df["amount_inr"] > 0]

# checking KYC Status
df["kyc_status"] = (
    df["kyc_status"]
    .astype(str)
    .str.strip()
    .str.title()
)

print("\nKYC Status Distribution:")
print(df["kyc_status"].value_counts())

valid_kyc = ["Verified", "Pending", "Rejected"]

invalid_kyc = df[
    ~df["kyc_status"].isin(valid_kyc)
]

print(f"\nInvalid KYC Records: {len(invalid_kyc)}")

if len(invalid_kyc) > 0:
    print("\nUnexpected KYC Values:")
    print(invalid_kyc["kyc_status"].value_counts())

# removing exact duplicates
before = len(df)

df = df.drop_duplicates()

after = len(df)

print(f"\nDuplicates Removed: {before - after}")



df = df.sort_values(
    ["investor_id", "transaction_date"]
).reset_index(drop=True)



print("\n===== FINAL SUMMARY =====")
print("Shape:", df.shape)

print("\nTransaction Types:")
print(df["transaction_type"].value_counts())

print("\nKYC Status:")
print(df["kyc_status"].value_counts())

print("\nAmount Statistics:")
print(df["amount_inr"].describe())



output_file = "investor_transactions_cleaned.csv"

df.to_csv(output_file, index=False)

print(f"\nCleaned file saved as: {output_file}")

Original Shape: (32778, 13)
Invalid Dates: 0

Invalid Transaction Types:
Series([], Name: count, dtype: int64)

Transactions with amount <= 0: 0

KYC Status Distribution:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Invalid KYC Records: 0

Duplicates Removed: 0

===== FINAL SUMMARY =====
Shape: (32778, 13)

Transaction Types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC Status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Amount Statistics:
count     32778.000000
mean     107437.318628
std      150415.905084
min         400.000000
25%        3153.000000
50%       17782.500000
75%      189324.250000
max      597498.000000
Name: amount_inr, dtype: float64

Cleaned file saved as: investor_transactions_cleaned.csv


schema_performance cleaning

In [3]:
import pandas as pd
import numpy as np

# Load file
df = pd.read_csv("/content/07_scheme_performance.csv")

print("Original Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())



return_cols = [
    col for col in df.columns
    if "return" in col.lower()
]

print("\nReturn Columns Detected:")
print(return_cols)



for col in return_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Checking non-numeric values
print("\nMissing/Invalid Return Values:")
print(df[return_cols].isnull().sum())



anomalies = pd.DataFrame()

for col in return_cols:
    temp = df[
        (df[col] < -100) |
        (df[col] > 200)
    ].copy()

    if len(temp) > 0:
        temp["anomaly_column"] = col
        anomalies = pd.concat([anomalies, temp])

print(f"\nAnomalous Return Records: {len(anomalies)}")

if len(anomalies) > 0:
    display(
        anomalies[
            ["anomaly_column"] +
            [c for c in df.columns if c != "anomaly_column"][:5]
        ].head(20)
    )




expense_col = None

for col in df.columns:
    if "expense" in col.lower():
        expense_col = col
        break

if expense_col:
    print(f"\nExpense Ratio Column Found: {expense_col}")

    df[expense_col] = pd.to_numeric(
        df[expense_col],
        errors="coerce"
    )

    invalid_expense = df[
        (df[expense_col] < 0.1) |
        (df[expense_col] > 2.5)
    ]

    print(
        f"Expense Ratio Outside 0.1% - 2.5%: "
        f"{len(invalid_expense)}"
    )

    if len(invalid_expense) > 0:
        display(
            invalid_expense[
                [expense_col] +
                [c for c in df.columns if c != expense_col][:5]
            ].head(20)
        )

    print("\nExpense Ratio Statistics:")
    print(df[expense_col].describe())

else:
    print("\nNo expense ratio column found.")



# missing values summary

print("\nMissing Values:")
print(df.isnull().sum())

#duplicate checking
duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicates}")


output_file = "scheme_performance_cleaned.csv"

df.to_csv(output_file, index=False)

print(f"\nCleaned file saved as: {output_file}")

Original Shape: (40, 19)

Columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

Return Columns Detected:
['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']

Missing/Invalid Return Values:
return_1yr_pct    0
return_3yr_pct    0
return_5yr_pct    0
dtype: int64

Anomalous Return Records: 0

Expense Ratio Column Found: expense_ratio_pct
Expense Ratio Outside 0.1% - 2.5%: 0

Expense Ratio Statistics:
count    40.000000
mean      1.237000
std       0.386584
min       0.550000
25%       0.787500
50%       1.425000
75%       1.540000
max       1.640000
Name: expense_ratio_pct, dtype: float64

Missing Values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pc

portfolio_holdings cleaning

In [9]:
import pandas as pd

file_name = "/content/09_portfolio_holdings.csv"
df_holdings = pd.read_csv(file_name)

print("Before Cleaning:")
print(df_holdings.info())

str_cols = df_holdings.select_dtypes(include=['object']).columns
for col in str_cols:
    df_holdings[col] = df_holdings[col].astype(str).str.strip()

df_holdings['portfolio_date'] = pd.to_datetime(df_holdings['portfolio_date'], errors='coerce')

df_holdings.drop_duplicates(inplace=True)

df_holdings.sort_values(by=['portfolio_date', 'amfi_code', 'weight_pct'], ascending=[True, True, False], inplace=True)
df_holdings.reset_index(drop=True, inplace=True)

cleaned_file_name = "cleaned_09_portfolio_holdings.csv"
df_holdings.to_csv(cleaned_file_name, index=False)

print("\nAfter Cleaning:")
print(df_holdings.info())
print(f"Cleaned file saved as: {cleaned_file_name}")

Before Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   amfi_code          322 non-null    int64  
 1   stock_symbol       322 non-null    object 
 2   stock_name         322 non-null    object 
 3   sector             322 non-null    object 
 4   weight_pct         322 non-null    float64
 5   market_value_cr    322 non-null    float64
 6   current_price_inr  322 non-null    float64
 7   portfolio_date     322 non-null    object 
dtypes: float64(3), int64(1), object(4)
memory usage: 20.3+ KB
None

After Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   amfi_code          322 non-null    int64         
 1   stock_symbol       322 non-null    objec

benchmark_indices cleaning

In [10]:
import pandas as pd

file_name = "/content/10_benchmark_indices.csv"
df_benchmarks = pd.read_csv(file_name)

print("Before Cleaning:")
print(df_benchmarks.info())

str_cols = df_benchmarks.select_dtypes(include=['object']).columns
for col in str_cols:
    df_benchmarks[col] = df_benchmarks[col].astype(str).str.strip()

df_benchmarks['date'] = pd.to_datetime(df_benchmarks['date'], errors='coerce')

df_benchmarks.drop_duplicates(inplace=True)

df_benchmarks.sort_values(by=['date', 'index_name'], inplace=True)
df_benchmarks.reset_index(drop=True, inplace=True)

cleaned_file_name = "cleaned_10_benchmark_indices.csv"
df_benchmarks.to_csv(cleaned_file_name, index=False)

print("\nAfter Cleaning:")
print(df_benchmarks.info())
print(f"Cleaned file saved as: {cleaned_file_name}")

Before Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8050 entries, 0 to 8049
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         8050 non-null   object 
 1   index_name   8050 non-null   object 
 2   close_value  8050 non-null   float64
dtypes: float64(1), object(2)
memory usage: 188.8+ KB
None

After Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8050 entries, 0 to 8049
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         8050 non-null   datetime64[ns]
 1   index_name   8050 non-null   object        
 2   close_value  8050 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 188.8+ KB
None
Cleaned file saved as: cleaned_10_benchmark_indices.csv
